# Chương 3 - Tiền xử lý và khám phá dữ liệu văn bản

Notebook thực hành quy trình tiền xử lý văn bản tiếng Việt theo nội dung Chương 3.

Quy trình: chữ thường → URL → emoji/emoticon → viết tắt → sửa chính tả → tách câu → xóa dấu câu/chữ số → tách từ → stop words → thống kê và trực quan hóa.

## 1. Cài đặt thư viện

Chỉ sử dụng các thư viện phục vụ trực tiếp cho nội dung bài học. Nếu đã cài đặt rồi, có thể bỏ qua cell này.

In [ ]:
%pip install underthesea emoji wordcloud matplotlib

## 2. Import thư viện và thiết lập đường dẫn

Đổi `INPUT_FILE` thành `data_raw_2.txt` nếu muốn kiểm tra bộ dữ liệu thứ hai.

In [ ]:
import re
import csv
from pathlib import Path
from collections import Counter

import emoji
import matplotlib.pyplot as plt
from underthesea import word_tokenize
from wordcloud import WordCloud

INPUT_FILE = Path('Cau1_raw.txt')
STOPWORDS_FILE = Path('vietnamese-stopwords.txt')
OUTPUT_DIR = Path('ket_qua_chapter3')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_PREFIX = INPUT_FILE.stem
TOP_N = 20

## 3. Đọc dữ liệu raw

Dữ liệu được đọc bằng UTF-8 để giữ tiếng Việt.

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Không tìm thấy file: {INPUT_FILE}')

raw_text = INPUT_FILE.read_text(encoding='utf-8-sig')
if not raw_text.strip():
    raise ValueError('File dữ liệu đang rỗng.')

print(f'Độ dài văn bản: {len(raw_text)} ký tự')
print(raw_text[:1000])

## 4. Chuẩn bị từ viết tắt, lỗi chính tả và emoticon

In [ ]:
ABBREVIATIONS = {
    'nckh': 'nghiên_cứu_khoa_học',
    'nntn': 'ngôn_ngữ_tự_nhiên',
    'xhcn': 'xã_hội_chủ_nghĩa',
    'cntt': 'công_nghệ_thông_tin',
    'ai': 'trí_tuệ_nhân_tạo',
    'ml': 'học_máy',
    'nlp': 'xử_lý_ngôn_ngữ_tự_nhiên',
    'ko': 'không', 'dc': 'được', 'mik': 'mình',
    'sp': 'sản_phẩm', 'ib': 'nhắn_tin'
}

SPELLING_CORRECTIONS = {
    'sữ dụng': 'sử dụng',
    'trãi nghiệm': 'trải nghiệm',
    'sữ lý': 'xử lý',
    'hiêu quả': 'hiệu quả'
}

EMOTICONS = {
    ':-)': 'vui', ':)': 'vui', ':-d': 'rất_vui', ':d': 'rất_vui',
    ':-(': 'buồn', ':(': 'buồn', ';-)': 'nháy_mắt', ';)': 'nháy_mắt',
    ':-p': 'trêu_đùa', ':p': 'trêu_đùa'
}

## 5. Các hàm tiền xử lý

Mỗi hàm tương ứng với một bước trong slide. Tách riêng các hàm giúp dễ kiểm tra kết quả từng bước.

In [ ]:
def remove_urls(text):
    return re.sub(r'(?:https?://|www\.)\S+', ' ', text, flags=re.IGNORECASE)

def replace_emoticons(text):
    pattern = re.compile('|'.join(re.escape(k) for k in EMOTICONS), re.IGNORECASE)
    return pattern.sub(lambda m: f' {EMOTICONS[m.group().lower()]} ', text)

def replace_emojis(text):
    text = emoji.demojize(text, language='en')
    return text.replace(':', ' ').replace('_', ' ')

def expand_abbreviations(text):
    pattern = re.compile(r'\b(' + '|'.join(ABBREVIATIONS) + r')\b', re.IGNORECASE)
    return pattern.sub(lambda m: f' {ABBREVIATIONS[m.group().lower()]} ', text)

def correct_spelling(text):
    for wrong, right in SPELLING_CORRECTIONS.items():
        text = re.sub(rf'\b{re.escape(wrong)}\b', right, text, flags=re.IGNORECASE)
    return text

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', text) if s.strip()]

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE).replace('_', ' ')

def remove_numbers(text):
    return re.sub(r'\d+', ' ', text)

def normalize_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

def normalize_tokens(tokens):
    return [
        token.lower().strip()
        for token in tokens
        if len(token.strip()) > 1 and re.search(r'[a-zA-ZÀ-ỹ]', token)
    ]

## 6. Thực hiện tiền xử lý theo đúng thứ tự

Tách câu được thực hiện trước khi xóa dấu câu để vẫn giữ được các dấu `.`, `?`, `!` dùng làm ranh giới câu.

In [ ]:
text = raw_text.lower()
text = remove_urls(text)
text = replace_emoticons(text)
text = replace_emojis(text)
text = expand_abbreviations(text)
text = correct_spelling(text)

sentences = split_sentences(text)
text_clean = ' '.join(sentences)
text_clean = remove_numbers(text_clean)
text_clean = remove_punctuation(text_clean)
text_clean = normalize_spaces(text_clean)

print(f'Số câu sau khi tách: {len(sentences)}')
print(text_clean[:1000])

## 7. Tách từ và loại bỏ Stop Words

Sử dụng `Underthesea` vì dữ liệu là tiếng Việt.

In [ ]:
if STOPWORDS_FILE.exists():
    stopwords = {
        re.sub(r'\s+', '_', line.strip().lower())
        for line in STOPWORDS_FILE.read_text(encoding='utf-8-sig').splitlines()
        if line.strip()
    }
else:
    stopwords = {'và', 'là', 'của', 'có', 'cho', 'với', 'một', 'những', 'các'}

tokens_before_stopwords = word_tokenize(text_clean, format='text').split()
tokens = [
    token for token in tokens_before_stopwords
    if token.lower() not in stopwords
]
tokens = normalize_tokens(tokens)

print(f'Token trước khi xóa stop words: {len(tokens_before_stopwords)}')
print(f'Token sau khi xóa stop words: {len(tokens)}')
print(tokens[:50])

## 8. Lưu văn bản đã tiền xử lý

Lưu dữ liệu sau xử lý để kiểm tra lại trước khi vector hóa hoặc xây dựng mô hình.

In [ ]:
cleaned_text = ' '.join(tokens)
cleaned_file = OUTPUT_DIR / f'{OUTPUT_PREFIX}_txl.txt'
cleaned_file.write_text(cleaned_text, encoding='utf-8')
print(f'Đã lưu: {cleaned_file}')

## 9. Thống kê tần suất từ

Tần suất giúp khám phá những từ xuất hiện nhiều nhất trong tập dữ liệu.

In [ ]:
frequency = Counter(tokens)
top_frequency = frequency.most_common(TOP_N)

with open(OUTPUT_DIR / f'{OUTPUT_PREFIX}_frequency.csv', 'w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerow(['word', 'frequency'])
    writer.writerows(top_frequency)

for word, count in top_frequency:
    print(f'{word}: {count}')

## 10. Trực quan hóa tần suất bằng biểu đồ

In [ ]:
words = [item[0] for item in top_frequency]
counts = [item[1] for item in top_frequency]

plt.figure(figsize=(12, 6))
plt.bar(words, counts)
plt.title('Tần suất xuất hiện của các từ')
plt.xlabel('Từ')
plt.ylabel('Tần suất')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'{OUTPUT_PREFIX}_frequency.png', dpi=300)
plt.show()

## 11. Word Cloud

Word Cloud biểu diễn các từ xuất hiện nhiều bằng kích thước lớn hơn.

In [ ]:
font_path = 'C:/Windows/Fonts/arial.ttf'
wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    collocations=False,
    font_path=font_path if Path(font_path).exists() else None
).generate_from_frequencies(frequency)

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'{OUTPUT_PREFIX}_wordcloud.png', dpi=300)
plt.show()

## 12. Kết luận

- Dữ liệu raw đã được làm sạch và chuẩn hóa.
- Văn bản đã được tách câu, tách từ và loại bỏ stop words.
- Kết quả đã được lưu để kiểm tra lại.
- Tần suất từ, biểu đồ và Word Cloud giúp khám phá dữ liệu văn bản.

Stemming và Lemmatization theo WordNet không áp dụng cho dữ liệu tiếng Việt trong notebook này; phần đó được minh họa trong slide bằng ví dụ tiếng Anh.